In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, abs as sabs
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

spark = SparkSession.builder.appName("Tugas12-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

26/11/19 07:48:45 WARN Utils: Your hostname, tarin resolves to a loopback address: 127.0.1.1; using 10.90.70.61 instead (on interface wlp0s20f3)
26/11/19 07:48:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 07:48:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 07:48:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



In [3]:
#a. membangun pipeline
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

fe_stages = [idx_menu, idx_cabang, idx_waktu, ohe, assembler]
pipeline_lr = Pipeline(stages=fe_stages + [lr])
print("Pipeline Logistic Regression terdefinisi (6 tahap).")

Pipeline Logistic Regression terdefinisi (6 tahap).


In [4]:
#b. pelatihan dan evaluasi
train, test = df.randomSplit([0.8, 0.2], seed=77)

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [5]:
#c. model pembanding
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)
pipeline_rf = Pipeline(stages=fe_stages + [rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== Perbandingan Hasil ===")
print(f"{'Model':22s}{'Accuracy':>10s}{'F1':>10s}{'AUC':>10s}")
print(f"{'Logistic Regression':22s}{acc_lr:>10.4f}{f1_lr:>10.4f}{auc_lr:>10.4f}")
print(f"{'Random Forest':22s}{acc_rf:>10.4f}{f1_rf:>10.4f}{auc_rf:>10.4f}")

print("\nConfusion Matrix Random Forest:")
pred_rf.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Perbandingan Hasil ===
Model                   Accuracy        F1       AUC
Logistic Regression       0.7929    0.7926    0.8650
Random Forest             0.7929    0.7927    0.8496

Confusion Matrix Random Forest:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  256|
|             0|       1.0|   58|
|             1|       0.0|   64|
|             1|       1.0|  211|
+--------------+----------+-----+



In [6]:
#d. analisis kesalahan model
# Pilih model terbaik berdasarkan AUC
if auc_rf > auc_lr:
    nama_terbaik, pred_terbaik, model_terbaik = "Random Forest", pred_rf, model_rf
else:
    nama_terbaik, pred_terbaik, model_terbaik = "Logistic Regression", pred_lr, model_lr
print("Model terbaik:", nama_terbaik)

salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print("Total prediksi salah:", salah.count(), "dari", pred_terbaik.count())

# 10 baris yang salah diprediksi
salah.select("nama_menu", "cabang", "waktu_pesan", "qty",
             "waktu_tunggu_menit", "ulasan_positif", "prediction").show(10)

# Pola: sebaran kesalahan menurut waktu tunggu
print("Sebaran kesalahan berdasarkan waktu_tunggu_menit:")
salah.groupBy("waktu_tunggu_menit").count().orderBy("waktu_tunggu_menit").show(25)

# Perbandingan tingkat kesalahan per cabang & menu (untuk cek apakah ada pola lain)
total = pred_terbaik.groupBy("cabang").count().withColumnRenamed("count", "total")
err = salah.groupBy("cabang").count().withColumnRenamed("count", "salah")
total.join(err, "cabang").withColumn("error_rate", col("salah") / col("total")).show()

Model terbaik: Logistic Regression
Total prediksi salah: 122 dari 589
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0|
|Americano|Semarang|      Ma

In [7]:
#bahan untuk e
# Nama fitur sesuai urutan di VectorAssembler
meta = pred_terbaik.schema["features"].metadata["ml_attr"]["attrs"]
nama_fitur = [a["name"] for k in meta for a in meta[k]]

if nama_terbaik == "Random Forest":
    nilai = model_terbaik.stages[-1].featureImportances.toArray()
else:
    nilai = model_terbaik.stages[-1].coefficients.toArray()

for n, v in sorted(zip(nama_fitur, nilai), key=lambda x: -abs(x[1]))[:5]:
    print(f"{n:30s} {v:.4f}")

cabang_vec_Semarang            0.2530
waktu_vec_Sore                 -0.2220
waktu_vec_Pagi                 0.1857
menu_vec_Cappuccino            -0.1443
qty                            0.1171


Berdasarkan model klasifikasi yang dibangun, fitur yang paling berpengaruh terhadap peluang ulasan positif adalah waktu_tunggu_menit. Fitur lain seperti menu, cabang, dan waktu pemesanan hampir tak mempengaruhi prediksi dan model paling sering salah pada pesanan denan waktu tunggu sekitar 10-14 menit, yaitu zona abu abu dimana pelanggan bisa puas atau kecewa. REkomendasi konkreit untuk manajemen KOpi Nusantara yaitu yang pertama, tetapkan target waktu penyajian maksimal sekitar 10 menit untuk semua cabang, karena semakin lama pelanggan menunggu semakin kecil pula peluang ulasan psitif. Lalu yang edua, siapkan sistem peringatan dini, misalnya notifikasi kepada barista atau supervisor ketika pesanan mendekati waktu 10 menit. Agar pesanan bisa diprioritaskan sebelum pelanggan kecewa. KEmudian yang ketiga, fokuskan pembenahan pada proses dapur dan antrean dengan contoh penambahan staff di waktu sibuk juga penyederhanaan prosedur penyajian. BUkan pada perubahan menu atau pemilihan cabang, sebab data tidak menunjukkan perbedaan signifikan disana. SElanjutnya yang keempat, tawarkan kompensasi kecil seperti diskon atau minuman gratis pada pesanan yang terlambat untuk menyelamatkan kepuasan pelanggan. TErakhir, model ini sehrusnya dipantau dan dilatih ulang secara berkala dengan data bru agar rekomendasinya tetap relevan